In [1]:
# imports
import MDAnalysis as mda
import pandas as pd
import nglview as nv
from merge_structures import assign_segment_ids, identify_subaggregates, merge_structures


In [2]:
existing_segids = {}
protein_data = {}
for prot in ["hpl2", "lin13", "lin65", "met2"]:
    x = mda.Universe(f"input_structures/single_comp_structures/{prot}.pdb")
    u = x.copy()

    # Add unique segment IDs to differentiate between protein chains
    u = assign_segment_ids(u, prot, existing_segids)
    
    # assign known universe dimensions
    u.dimensions = [600,600,3000,90,90,90]

    # Analyze chains and clusters
    clusters = identify_subaggregates(u)    
    protein_data[prot] = {
        'universe': u,
        "n_atoms": len(u.atoms),
        'subaggregates': clusters,
        'rg_max': max([prot.atoms.radius_of_gyration() for prot in u.segments]),
        "n_proteins": len(u.segments), 
        "box_dimensions": u.dimensions
    }

proteins_df = pd.DataFrame.from_dict(protein_data, orient='index')
proteins_df

,universe,n_atoms,subaggregates,rg_max,n_proteins,box_dimensions
hpl2,<Universe with 14000 atoms>,14000,"[(<Atom 1: CA of type C of resname MET, resid ...",58.676363,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"
lin13,<Universe with 89920 atoms>,89920,"[(<Atom 1: CA of type C of resname MET, resid ...",139.430734,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"
lin65,<Universe with 29120 atoms>,29120,"[(<Atom 1: CA of type C of resname MET, resid ...",186.969647,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"
met2,<Universe with 14000 atoms>,14000,"[(<Atom 1: CA of type C of resname MET, resid ...",58.676363,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"


In [4]:
merged = merge_structures(proteins_df)

TypeError: range() takes no keyword arguments

In [ ]:
view = nv.show_mdanalysis(merged)
view

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 1. Identify your clusters
subaggregates = identify_subaggregates(u, cutoff=20.0)
n_clusters = len(subaggregates)

# 2. Generate a list of Hex colors from a colormap (e.g., 'turbo' or 'viridis')
# 'turbo' is great for distinct, high-contrast colors
cmap = plt.get_cmap('turbo', n_clusters)
hex_colors = [mcolors.to_hex(cmap(i)) for i in range(n_clusters)]

# 3. Create the view
view = nv.show_mdanalysis(u)
view.clear()

# 4. Apply the colors
for i, agg in enumerate(subaggregates):
    view.add_representation('spacefill', 
                            selection=agg.indices, 
                            color=hex_colors[i],
                            opacity=0.9)

# 5. Optional: Add the unit cell box to see your 600A limits
view.add_unitcell()

print(f"Generated {n_clusters} unique colors for your subaggregates.")
view

Generated 29 unique colors for your subaggregates.


NGLWidget()